In [6]:
import numpy as np


input_text = "hello world"

input_text_embeddings = [[1,2,3,4], 
                         [2,3,4,5]]
input_text_embeddings = np.array(input_text_embeddings)

def get_positional_encoding(x):
    """
    Get positional encoding for the input text embeddings.
    """
    # Assuming x is a 2D numpy array with shape (batch_size, embedding_dim)
    batch_size, embedding_dim = x.shape
    positional_encoding = np.zeros((batch_size, embedding_dim))

    for pos in range(batch_size):
        for i in range(embedding_dim):
            if i % 2 == 0:
                positional_encoding[pos, i] = np.sin(pos / (10000 ** (2*i / embedding_dim)))
            else:
                positional_encoding[pos, i] = np.cos(pos / (10000 ** (2*i / embedding_dim)))

    return positional_encoding

def add_positional_encoding(embeddings):
    """
    Add positional encoding to the input text embeddings.
    """
    positional_encoding = get_positional_encoding(embeddings)
    return embeddings + positional_encoding


print("Input text:", input_text)
print("Input text embeddings:", input_text_embeddings)
    
# Get positional encoding
positional_encoding = get_positional_encoding(input_text_embeddings)
print("Positional encoding:", positional_encoding)

# Add positional encoding to the input text embeddings
embeddings_with_positional_encoding = add_positional_encoding(input_text_embeddings)
print("Embeddings with positional encoding:", embeddings_with_positional_encoding)

Input text: hello world
Input text embeddings: [[1 2 3 4]
 [2 3 4 5]]
Positional encoding: [[0.00000000e+00 1.00000000e+00 0.00000000e+00 1.00000000e+00]
 [8.41470985e-01 9.99950000e-01 9.99999998e-05 1.00000000e+00]]
Embeddings with positional encoding: [[1.         3.         3.         5.        ]
 [2.84147098 3.99995    4.0001     6.        ]]


In [7]:
WK1 = np.array([[1,0,1], [0,1,0], [1,0,1], [0,1,0]])
WV1 = np.array([[0,1,1], [1,0,0], [1,0,1], [0,1,0]])
WQ1 = np.array([[0,0,0], [1,1,0], [0,0,1], [1,0,0]])

WK2 = np.array([[0, 1, 1], [1, 0, 1], [1, 1, 0], [0, 1, 0]])
WV2 = np.array([[1, 0, 0], [0, 1, 1], [0, 0, 1], [1, 0, 0]])
WQ2 = np.array([[1, 0, 1], [0, 1, 0], [1, 0, 0], [0, 1, 1]])

In [26]:
K1 = embeddings_with_positional_encoding @ WK1
V1 = embeddings_with_positional_encoding @ WV1
Q1 = embeddings_with_positional_encoding @ WQ1
K2 = embeddings_with_positional_encoding @ WK2
V2 = embeddings_with_positional_encoding @ WV2
Q2 = embeddings_with_positional_encoding @ WQ2

K1, V1, Q1

(array([[4.        , 8.        , 4.        ],
        [6.84157098, 9.99995   , 6.84157098]]),
 array([[6.        , 6.        , 4.        ],
        [8.00005   , 8.84147098, 6.84157098]]),
 array([[8.     , 3.     , 3.     ],
        [9.99995, 3.99995, 4.0001 ]]))

In [27]:
scores1 = np.dot(Q1, K1.T) / np.sqrt(3)
scores1

array([[39.2598183 , 60.77023282],
       [50.80670822, 78.39356402]])

In [24]:
def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=1, keepdims=True)

"""
keepdims
False (default): dimension over which operation is performed are **removed**
True: axes are retained with size 1

a = np.array([[1, 2, 3],
              [4, 5, 6]])
np.sum(a, axis=1)                >> Output: array([ 6, 15])     >> shape: (2,)
np.sum(a, axis=1, keepdims=True) >> Output: array([[ 6], 
                                                   [15]])       >> shape: (2,1)
"""

softmax([[1,2], [3, 0.5]])

array([[0.26894142, 0.73105858],
       [0.92414182, 0.07585818]])

In [28]:
scores1 = softmax(scores1)
scores1

array([[4.55140699e-10, 1.00000000e+00],
       [1.04515512e-12, 1.00000000e+00]])

In [29]:
scores1 = scores1 @ V1
scores1

array([[8.00005   , 8.84147098, 6.84157098],
       [8.00005   , 8.84147098, 6.84157098]])

In [30]:
def attention(x, WQ, WK, WV):
    Q = x @ WQ
    K = x @ WK
    V = x @ WV

    score = Q @ K.T
    score = score / np.sqrt(3)
    score = softmax(score)
    score = score @ V
    return score

att1 = attention(embeddings_with_positional_encoding, WQ1, WK1, WV1)
att2 = attention(embeddings_with_positional_encoding, WQ2, WK2, WV2)

concat_att = np.concatenate([att1, att2], axis=1)
concat_att

array([[8.00005   , 8.84147098, 6.84157098, 8.84147098, 3.99995   ,
        8.00005   ],
       [8.00005   , 8.84147098, 6.84157098, 8.84147098, 3.99995   ,
        8.00005   ]])

In [31]:
W_att =  np.array(
    [
        [0.79445237, 0.1081456, 0.27411536, 0.78394531],
        [0.29081936, -0.36187258, -0.32312791, -0.48530339],
        [-0.36702934, -0.76471963, -0.88058366, -1.73713022],
        [-0.02305587, -0.64315981, -0.68306653, -1.25393866],
        [0.29077448, -0.04121674, 0.01509932, 0.13149906],
        [0.57451867, -0.08895355, 0.02190485, 0.24535932],
    ]
)
Z = concat_att @ W_att
Z

array([[ 11.97128599, -14.12917589, -12.49224156, -18.50167966],
       [ 11.971286  , -14.12917589, -12.49224156, -18.50167966]])

In [37]:
def relu(x):
    return np.maximum(0, x)


def ffnn(x):
    W1 = np.random.rand(4, 8)
    B1 = np.random.rand(8)

    W2 = np.random.rand(8, 4)
    B2 = np.random.rand(4)

    print(W1, x, x.dot(W1))
    return relu(x.dot(W1) + B1).dot(W2) + B2


ffnn(Z)

[[0.59944209 0.10591294 0.22144651 0.812756   0.89782019 0.63581587
  0.85429519 0.54731625]
 [0.16489605 0.77728692 0.78209779 0.41065729 0.22435772 0.18257357
  0.72843745 0.41852946]
 [0.41468962 0.62041577 0.46542328 0.62454834 0.9261104  0.36315781
  0.88021268 0.88752152]
 [0.75609616 0.74011779 0.88185864 0.76219121 0.33519607 0.56376439
  0.27731414 0.96249655]] [[ 11.97128599 -14.12917589 -12.49224156 -18.50167966]
 [ 11.971286   -14.12917589 -12.49224156 -18.50167966]] [[-14.32320443 -31.15831551 -30.52944381 -17.97634073 -10.19281242
   -9.93532354 -16.19181564 -28.25633307]
 [-14.32320443 -31.15831551 -30.52944381 -17.97634073 -10.19281242
   -9.93532354 -16.19181564 -28.25633308]]


array([[0.30060575, 0.72531008, 0.99526867, 0.55242738],
       [0.30060575, 0.72531008, 0.99526867, 0.55242738]])

In [ ]:
## ALL Together
d_embedding = 4
n_encoder_block = 6
n_att_head = 8
d_key = d_value = d_query = 3
d_feedforward = 8

def softmax(x):
    return np.exp(x) / np.sum(np.exp(x), axis=1, keepdims=True)
def relu(x):
    return np.maximum(0, x)

def attention_layer(x, WQ, WK, WV):
    Q = x @ WQ
    K = x @ WK
    V = x @ WV

    score = Q @ K.T
    score = score / np.sqrt(d_value)
    score = softmax(score)
    score = score @ V
    return score

def multi_head_attention(x, WQs, WKs, WVs, n_att_head):
    all_attentions = [
        np.concatenate([attention_layer(x, WQ, WK, WV) for (WQ, WK, WV) in zip(WQs, WKs, WVs)], axis=1)
    ]
    W_att = np.random.rand(n_att_head*d_value, d_embedding)
    return all_attentions @ W_att

def ffnn_layer(x, W1, B1, W2, B2):
    return relu(x.dot(W1) + B1).dot(W2) + B2

def layer_norm(x, epsilon=1e-6):
    mean = x.mean(axis=-1, keepdims=True) # learn more about axis=-1
    std = x.std(axis=-1, keepdims=True)
    return (x-mean) / (std + epsilon)

def encoder_block(x, n_att_head=8):
    WQs = [ np.random.rand(d_embedding, d_query) for _ in range(n_att_head)]
    WKs = [ np.random.rand(d_embedding, d_key) for _ in range(n_att_head)]
    WVs = [ np.random.rand(d_embedding, d_value) for _ in range(n_att_head)]

    W1 = np.random.rand(d_embedding, d_feedforward)
    B1 = np.random.rand(d_feedforward)
    W2 = np.random.rand(d_feedforward, d_embedding)
    B2 = np.random.rand(d_embedding)

    
    Z = multi_head_attention(x, WQs, WKs, WVs, n_att_head)
    Z = layer_norm(Z+x)
    output = ffnn_layer(Z, W1, B1, W2, B2)
    Z = layer_norm(Z+output)
    return x

def encoder(x, n_encoder_block=6):
    for _ in range(n_encoder_block):
        x = encoder_block(x)
    return x
output = encoder(embeddings_with_positional_encoding)